In [ ]:
"""
=============================================================================
NOTEBOOK 02: MODEL COMPARISON - BASE vs AMPLIADO
=============================================================================
Proyecto: Detección de Insider Trading en el Congreso de EE.UU.
Maestría en Economía - UdeSA

PREGUNTA DE INVESTIGACIÓN:
¿Los trades de congresistas contienen información que mejora la predicción 
de retornos de acciones más allá de la información pública de mercado?

DISEÑO EXPERIMENTAL:
  Modelo Base:     ret_future ~ f(FEATURES_MERCADO)
  Modelo Ampliado: ret_future ~ f(FEATURES_MERCADO + FEATURES_CONGRESO)
  
  Test: Si R²_OOS_ampliado > R²_OOS_base → Congresistas tienen información

METODOLOGÍA:
  - Expanding window (sin look-ahead bias)
  - Out-of-sample R² (Campbell-Thompson 2008)
  - Clark-West test para modelos anidados
  - Múltiples algoritmos para robustez

=============================================================================
"""

import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')

# ML
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from scipy import stats

# Plotting
import matplotlib.pyplot as plt

# Seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# =============================================================================
# 0. CONFIGURACIÓN
# =============================================================================

os.chdir('C:/Users/sebib/Documents/GitHub/US_Congress')
print(f"Directorio de trabajo: {os.getcwd()}")

INPUT_PANEL = 'data/prediction_bases/panel_final_stock_month.parquet'
INPUT_FEATURES = 'data/prediction_bases/feature_sets.json'
OUTPUT_DIR = 'outputs/Model_Comparison'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*70)
print("NOTEBOOK 02: MODEL COMPARISON - BASE vs AMPLIADO")
print("="*70)

# =============================================================================
# 1. CARGAR DATOS
# =============================================================================
print("\n[1] CARGANDO DATOS...")

panel = pd.read_parquet(INPUT_PANEL)
print(f"    Observaciones: {len(panel):,}")
print(f"    Variables: {len(panel.columns)}")

# Cargar feature sets
with open(INPUT_FEATURES, 'r') as f:
    feature_sets = json.load(f)

print(f"    Features mercado: {len(feature_sets['market'])}")
print(f"    Features congreso: {len(feature_sets['congress'])}")

# =============================================================================
# 2. PREPROCESAMIENTO
# =============================================================================
print("\n[2] PREPROCESAMIENTO...")

# --- 2.1 Definir variable objetivo ---
TARGET = 'ret_future_1m'

# --- 2.2 Winsorizar outliers ---
print("    2.1 Winzorizando outliers...")

def winsorize(series, limits=(0.01, 0.99)):
    """Winsoriza una serie al percentil especificado."""
    lower = series.quantile(limits[0])
    upper = series.quantile(limits[1])
    return series.clip(lower=lower, upper=upper)

# Winsorizar target
panel[TARGET] = winsorize(panel[TARGET])
print(f"        Target winsorizado: [{panel[TARGET].min():.4f}, {panel[TARGET].max():.4f}]")

# Winsorizar features numéricas
numeric_cols = panel.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if col != TARGET and col not in ['month']:
        panel[col] = winsorize(panel[col])

# --- 2.3 Manejar NaNs ---
print("    2.2 Manejando NaNs...")
print(f"        NaNs antes: {panel.isnull().sum().sum():,}")

# Para features, rellenar con mediana
for col in feature_sets['market'] + feature_sets['congress']:
    if col in panel.columns:
        panel[col] = panel[col].fillna(panel[col].median())

# Eliminar filas sin target
panel = panel.dropna(subset=[TARGET])
print(f"        Observaciones finales: {len(panel):,}")

# --- 2.4 Convertir month a datetime para ordenar ---
panel['month_dt'] = panel['month'].astype(str).apply(lambda x: pd.Period(x, freq='M').to_timestamp())
panel = panel.sort_values(['month_dt', 'ticker'])

print(f"        Período: {panel['month'].min()} a {panel['month'].max()}")

# =============================================================================
# 3. DEFINIR FEATURE SETS
# =============================================================================
print("\n[3] DEFINIENDO FEATURE SETS...")

# Features de mercado (disponibles y sin NaN)
FEATURES_MARKET = [f for f in feature_sets['market'] if f in panel.columns]
FEATURES_MARKET = [f for f in FEATURES_MARKET if panel[f].notna().sum() > len(panel)*0.5]

# Features de congreso
FEATURES_CONGRESS = [f for f in feature_sets['congress'] if f in panel.columns]

# Conjunto completo
FEATURES_ALL = FEATURES_MARKET + FEATURES_CONGRESS

print(f"    Features mercado (modelo base): {len(FEATURES_MARKET)}")
print(f"    Features congreso (adicionales): {len(FEATURES_CONGRESS)}")
print(f"    Features totales (modelo ampliado): {len(FEATURES_ALL)}")

# Verificar que no hay NaNs en features
for f in FEATURES_ALL:
    if panel[f].isna().any():
        panel[f] = panel[f].fillna(panel[f].median())

# =============================================================================
# 4. FUNCIONES DE EVALUACIÓN
# =============================================================================
print("\n[4] DEFINIENDO FUNCIONES DE EVALUACIÓN...")

def oos_r2(y_true, y_pred, y_benchmark):
    """
    Out-of-sample R² (Campbell-Thompson 2008).
    
    R²_OOS = 1 - (SSE_model / SSE_benchmark)
    
    Positivo: modelo mejor que benchmark
    Cero: igual al benchmark
    Negativo: peor que benchmark
    """
    sse_model = np.sum((y_true - y_pred) ** 2)
    sse_bench = np.sum((y_true - y_benchmark) ** 2)
    
    if sse_bench == 0:
        return np.nan
    
    return 1 - (sse_model / sse_bench)


def clark_west_test(y_true, y_pred_restricted, y_pred_unrestricted):
    """
    Clark-West (2007) test para modelos anidados.
    
    H0: Modelo restringido es mejor o igual
    H1: Modelo no restringido (ampliado) es mejor
    
    Retorna: (estadístico, p-valor)
    """
    e1 = y_true - y_pred_restricted
    e2 = y_true - y_pred_unrestricted
    
    # MSPE adjustment
    adj = (y_pred_restricted - y_pred_unrestricted) ** 2
    f = e1**2 - (e2**2 - adj)
    
    n = len(f)
    t_stat = np.mean(f) / (np.std(f, ddof=1) / np.sqrt(n))
    p_value = 1 - stats.norm.cdf(t_stat)  # One-sided
    
    return t_stat, p_value


def diebold_mariano_test(e1, e2):
    """
    Diebold-Mariano test para comparar dos modelos.
    
    H0: Igual precisión predictiva
    H1: Diferente precisión
    
    Positivo: modelo 2 mejor
    """
    d = e1**2 - e2**2
    
    n = len(d)
    d_mean = np.mean(d)
    d_var = np.var(d, ddof=1)
    
    if d_var == 0:
        return 0, 1
    
    dm_stat = d_mean / np.sqrt(d_var / n)
    p_value = 2 * (1 - stats.norm.cdf(abs(dm_stat)))
    
    return dm_stat, p_value

# =============================================================================
# 5. EXPANDING WINDOW FORECASTER
# =============================================================================
print("\n[5] CONFIGURANDO EXPANDING WINDOW...")

class ExpandingWindowForecaster:
    """
    Expanding window con evaluación out-of-sample.
    
    En cada tiempo t:
    1. Entrenar con datos [0, t-1]
    2. Predecir t
    3. Avanzar a t+1
    
    Esto evita look-ahead bias.
    """
    
    def __init__(self, min_train_periods=36):
        self.min_train = min_train_periods
    
    def forecast(self, data, target, features, model, scale=True):
        """Genera predicciones out-of-sample."""
        
        # Obtener meses únicos ordenados
        months = sorted(data['month_dt'].unique())
        
        if len(months) <= self.min_train:
            return np.array([]), np.array([]), []
        
        predictions = []
        actuals = []
        dates = []
        
        for t in range(self.min_train, len(months)):
            # Datos de entrenamiento: meses [0, t-1]
            train_months = months[:t]
            test_month = months[t]
            
            train_mask = data['month_dt'].isin(train_months)
            test_mask = data['month_dt'] == test_month
            
            train_data = data[train_mask]
            test_data = data[test_mask]
            
            if len(test_data) == 0:
                continue
            
            X_train = train_data[features].values
            y_train = train_data[target].values
            X_test = test_data[features].values
            y_test = test_data[target].values
            
            # Escalar
            if scale:
                scaler = StandardScaler()
                X_train = scaler.fit_transform(X_train)
                X_test = scaler.transform(X_test)
            
            # Entrenar y predecir
            try:
                model.fit(X_train, y_train)
                preds = model.predict(X_test)
            except Exception as e:
                preds = np.full(len(y_test), np.nan)
            
            predictions.extend(preds)
            actuals.extend(y_test)
            dates.extend([test_month] * len(y_test))
        
        return np.array(predictions), np.array(actuals), dates
    
    def historical_average(self, data, target):
        """Benchmark: promedio histórico expandido."""
        
        months = sorted(data['month_dt'].unique())
        
        predictions = []
        actuals = []
        
        for t in range(self.min_train, len(months)):
            train_months = months[:t]
            test_month = months[t]
            
            train_mask = data['month_dt'].isin(train_months)
            test_mask = data['month_dt'] == test_month
            
            hist_mean = data.loc[train_mask, target].mean()
            y_test = data.loc[test_mask, target].values
            
            predictions.extend([hist_mean] * len(y_test))
            actuals.extend(y_test)
        
        return np.array(predictions), np.array(actuals)

# Configurar
MIN_TRAIN_MONTHS = 36  # 3 años de entrenamiento mínimo
forecaster = ExpandingWindowForecaster(min_train_periods=MIN_TRAIN_MONTHS)

print(f"    Meses mínimos de entrenamiento: {MIN_TRAIN_MONTHS}")

# =============================================================================
# 6. DEFINIR MODELOS
# =============================================================================
print("\n[6] DEFINIENDO MODELOS...")

MODELS = {
    'OLS': LinearRegression(),
    'Ridge': RidgeCV(alphas=[0.001, 0.01, 0.1, 1, 10, 100]),
    'LASSO': LassoCV(cv=5, max_iter=10000, random_state=RANDOM_STATE),
    'ElasticNet': ElasticNetCV(cv=5, max_iter=10000, random_state=RANDOM_STATE),
    'RandomForest': RandomForestRegressor(
        n_estimators=100,
        max_depth=5,
        min_samples_leaf=20,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    'GradientBoosting': GradientBoostingRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        min_samples_leaf=20,
        random_state=RANDOM_STATE
    ),
}

print(f"    Modelos a evaluar: {list(MODELS.keys())}")

# =============================================================================
# 7. EJECUTAR COMPARACIÓN
# =============================================================================
print("\n[7] EJECUTANDO COMPARACIÓN DE MODELOS...")
print("    (Esto puede tomar varios minutos...)\n")

# Feature sets a comparar
FEATURE_SETS = {
    'Base (Mercado)': FEATURES_MARKET,
    'Ampliado (Mercado + Congreso)': FEATURES_ALL,
}

# Benchmark
print("    Calculando benchmark (historical average)...")
ha_pred, ha_actual = forecaster.historical_average(panel, TARGET)
print(f"    Benchmark: {len(ha_pred):,} predicciones\n")

# Almacenar resultados
results = []
all_forecasts = {'Historical Average': {'pred': ha_pred, 'actual': ha_actual}}

# Ejecutar cada combinación
for feat_name, features in FEATURE_SETS.items():
    print(f"  === {feat_name} ({len(features)} features) ===")
    
    for model_name, model in MODELS.items():
        print(f"      {model_name}...", end=" ")
        
        # Obtener predicciones
        pred, actual, dates = forecaster.forecast(
            panel, TARGET, features, model
        )
        
        if len(pred) == 0:
            print("SKIP (no hay suficientes datos)")
            continue
        
        # Alinear con benchmark
        min_len = min(len(pred), len(ha_pred))
        pred = pred[:min_len]
        actual = actual[:min_len]
        ha = ha_pred[:min_len]
        
        # Guardar forecasts
        key = f'{model_name}_{feat_name}'
        all_forecasts[key] = {'pred': pred, 'actual': actual}
        
        # Calcular métricas
        r2 = oos_r2(actual, pred, ha)
        mse = np.mean((actual - pred) ** 2)
        
        # DM test vs benchmark
        e_model = actual - pred
        e_ha = actual - ha
        dm_stat, dm_pval = diebold_mariano_test(e_ha, e_model)
        
        results.append({
            'Model': model_name,
            'Features': feat_name,
            'N_features': len(features),
            'N_predictions': len(pred),
            'OOS_R2': r2,
            'OOS_R2_pct': r2 * 100,
            'MSE': mse,
            'RMSE': np.sqrt(mse),
            'DM_stat': dm_stat,
            'DM_pval': dm_pval,
        })
        
        print(f"R²_OOS = {r2*100:.3f}%")

results_df = pd.DataFrame(results)

# =============================================================================
# 8. RESULTADOS PRINCIPALES
# =============================================================================
print("\n" + "="*70)
print("RESULTADOS PRINCIPALES")
print("="*70)

# Tabla de resultados
print("\n--- Out-of-Sample R² por Modelo y Feature Set ---\n")
pivot = results_df.pivot(index='Model', columns='Features', values='OOS_R2_pct')
pivot['Mejora (pp)'] = pivot['Ampliado (Mercado + Congreso)'] - pivot['Base (Mercado)']
pivot = pivot.sort_values('Mejora (pp)', ascending=False)
print(pivot.round(4).to_string())

# =============================================================================
# 9. CLARK-WEST TEST
# =============================================================================
print("\n" + "="*70)
print("CLARK-WEST TEST: ¿CONGRESO AGREGA VALOR PREDICTIVO?")
print("="*70)
print("\nH0: Modelo base (mercado) es suficiente")
print("H1: Modelo ampliado (+ congreso) predice mejor\n")

cw_results = []

for model_name in MODELS.keys():
    key_base = f"{model_name}_Base (Mercado)"
    key_full = f"{model_name}_Ampliado (Mercado + Congreso)"
    
    if key_base in all_forecasts and key_full in all_forecasts:
        pred_base = all_forecasts[key_base]['pred']
        pred_full = all_forecasts[key_full]['pred']
        actual = all_forecasts[key_base]['actual']
        
        min_len = min(len(pred_base), len(pred_full), len(actual))
        
        cw_stat, cw_pval = clark_west_test(
            actual[:min_len],
            pred_base[:min_len],
            pred_full[:min_len]
        )
        
        sig = '***' if cw_pval < 0.01 else '**' if cw_pval < 0.05 else '*' if cw_pval < 0.10 else ''
        
        cw_results.append({
            'Model': model_name,
            'CW_stat': cw_stat,
            'CW_pval': cw_pval,
            'Significant': sig
        })
        
        print(f"  {model_name:20s}  CW = {cw_stat:7.3f}  p = {cw_pval:.4f} {sig}")

cw_df = pd.DataFrame(cw_results)

# =============================================================================
# 10. RESUMEN EJECUTIVO
# =============================================================================
print("\n" + "="*70)
print("RESUMEN EJECUTIVO")
print("="*70)

# Mejor modelo base
best_base = results_df[results_df['Features'] == 'Base (Mercado)'].sort_values('OOS_R2', ascending=False).iloc[0]

# Mejor modelo ampliado
best_full = results_df[results_df['Features'] == 'Ampliado (Mercado + Congreso)'].sort_values('OOS_R2', ascending=False).iloc[0]

# Cuántos modelos mejoran
n_improve = (pivot['Mejora (pp)'] > 0).sum()
n_total = len(pivot)

# CW significativos
n_cw_sig = cw_df['Significant'].str.len().gt(0).sum() if len(cw_df) > 0 else 0

print(f"""
DATOS:
  Observaciones totales:     {len(panel):,}
  Predicciones out-of-sample: {best_base['N_predictions']:,}
  Features mercado (base):   {len(FEATURES_MARKET)}
  Features congreso:         {len(FEATURES_CONGRESS)}

MEJOR MODELO BASE (solo mercado):
  Modelo: {best_base['Model']}
  R² OOS: {best_base['OOS_R2_pct']:.4f}%

MEJOR MODELO AMPLIADO (mercado + congreso):
  Modelo: {best_full['Model']}
  R² OOS: {best_full['OOS_R2_pct']:.4f}%

COMPARACIÓN:
  Modelos que mejoran con congreso: {n_improve}/{n_total}
  Tests Clark-West significativos:  {n_cw_sig}/{n_total}

INTERPRETACIÓN:
""")

avg_improvement = pivot['Mejora (pp)'].mean()
if avg_improvement > 0:
    print(f"  ✅ En promedio, agregar variables de congreso MEJORA la predicción")
    print(f"     en {avg_improvement:.4f} puntos porcentuales de R² OOS.")
else:
    print(f"  ❌ En promedio, agregar variables de congreso NO mejora la predicción.")

if n_cw_sig > 0:
    print(f"\n  ✅ {n_cw_sig} modelos muestran mejora ESTADÍSTICAMENTE SIGNIFICATIVA")
    print(f"     según el test de Clark-West.")
else:
    print(f"\n  ⚠️ Ningún modelo muestra mejora estadísticamente significativa.")

# =============================================================================
# 11. GUARDAR RESULTADOS
# =============================================================================
print("\n[11] GUARDANDO RESULTADOS...")

# Resultados principales
results_df.to_csv(os.path.join(OUTPUT_DIR, 'model_comparison_results.csv'), index=False)
print(f"    Guardado: {OUTPUT_DIR}/model_comparison_results.csv")

# Pivot table
pivot.to_csv(os.path.join(OUTPUT_DIR, 'oos_r2_comparison.csv'))
print(f"    Guardado: {OUTPUT_DIR}/oos_r2_comparison.csv")

# Clark-West
if len(cw_df) > 0:
    cw_df.to_csv(os.path.join(OUTPUT_DIR, 'clark_west_tests.csv'), index=False)
    print(f"    Guardado: {OUTPUT_DIR}/clark_west_tests.csv")

# =============================================================================
# 12. GRÁFICOS
# =============================================================================
print("\n[12] GENERANDO GRÁFICOS...")

# Configuración de estilo
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
    'figure.dpi': 120,
})

# --- Gráfico 1: Comparación R² OOS ---
fig, ax = plt.subplots(figsize=(10, 6))

models_order = pivot.index.tolist()
x = np.arange(len(models_order))
width = 0.35

bars1 = ax.bar(x - width/2, pivot['Base (Mercado)'], width, 
               label='Base (Mercado)', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, pivot['Ampliado (Mercado + Congreso)'], width,
               label='Ampliado (+Congreso)', color='coral', alpha=0.8)

ax.axhline(0, color='black', linewidth=0.8, linestyle='-')
ax.set_ylabel('Out-of-Sample R² (%)')
ax.set_xlabel('Modelo')
ax.set_title('Comparación: ¿Las Variables de Congreso Mejoran la Predicción?')
ax.set_xticks(x)
ax.set_xticklabels(models_order, rotation=45, ha='right')
ax.legend(loc='best')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'oos_r2_comparison.png'), dpi=150)
plt.savefig(os.path.join(OUTPUT_DIR, 'oos_r2_comparison.pdf'))
print(f"    Guardado: oos_r2_comparison.png/pdf")
plt.show()

# --- Gráfico 2: Mejora por modelo ---
fig, ax = plt.subplots(figsize=(8, 5))

improvements = pivot['Mejora (pp)'].sort_values()
colors = ['coral' if v > 0 else 'steelblue' for v in improvements]

bars = ax.barh(improvements.index, improvements.values, color=colors, alpha=0.8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Mejora en R² OOS (puntos porcentuales)')
ax.set_title('Mejora al Agregar Variables de Congreso')

# Etiquetas
for bar, val in zip(bars, improvements.values):
    x_pos = val + 0.001 if val >= 0 else val - 0.001
    ha = 'left' if val >= 0 else 'right'
    ax.annotate(f'{val:.3f}', xy=(x_pos, bar.get_y() + bar.get_height()/2),
                va='center', ha=ha, fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'improvement_by_model.png'), dpi=150)
plt.savefig(os.path.join(OUTPUT_DIR, 'improvement_by_model.pdf'))
print(f"    Guardado: improvement_by_model.png/pdf")
plt.show()

# =============================================================================
# FIN
# =============================================================================
print("\n" + "="*70)
print("✅ NOTEBOOK 02 COMPLETADO")
print("="*70)
print(f"\nResultados guardados en: {OUTPUT_DIR}/")

Directorio de trabajo: C:\Users\sebib\Documents\GitHub\US_Congress
NOTEBOOK 02: MODEL COMPARISON - BASE vs AMPLIADO

[1] CARGANDO DATOS...
    Observaciones: 32,169
    Variables: 84
    Features mercado: 29
    Features congreso: 41

[2] PREPROCESAMIENTO...
    2.1 Winzorizando outliers...
        Target winsorizado: [-0.0658, 0.0641]
    2.2 Manejando NaNs...
        NaNs antes: 15,331
        Observaciones finales: 32,169
        Período: 2012-07 a 2024-11

[3] DEFINIENDO FEATURE SETS...
    Features mercado (modelo base): 29
    Features congreso (adicionales): 41
    Features totales (modelo ampliado): 70

[4] DEFINIENDO FUNCIONES DE EVALUACIÓN...

[5] CONFIGURANDO EXPANDING WINDOW...
    Meses mínimos de entrenamiento: 36

[6] DEFINIENDO MODELOS...
    Modelos a evaluar: ['OLS', 'Ridge', 'LASSO', 'ElasticNet', 'RandomForest', 'GradientBoosting']

[7] EJECUTANDO COMPARACIÓN DE MODELOS...
    (Esto puede tomar varios minutos...)

    Calculando benchmark (historical average)...
   